# Crawling CampusNest (Streamlit) with politeness + extraction

This notebook demonstrates a **simple web crawler** for the CampusNest Streamlit site:

- https://campusnest.streamlit.app/

The crawler:

1. Starts from the home page
2. Fetches and honours `robots.txt` (basic rules + crawl-delay)
3. Uses a **queue (BFS)** to discover and visit internal links
4. Extracts tutor and listing data from visited pages
5. Saves the extracted datasets as CSV

Because Streamlit content is often rendered dynamically, we crawl using **Playwright** (headless Chromium) so we can discover links and extract the rendered DOM.


## 0) Install dependencies (first time only)


In [ ]:
!pip -q install playwright pandas requests
!playwright install chromium

## 1) Imports and configuration


In [ ]:
import re
import time
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from playwright.sync_api import sync_playwright

BASE_URL = "https://campusnest.streamlit.app/"
MAX_PAGES = 60          
DEFAULT_DELAY = 1.0     # seconds (used if robots.txt has no Crawl-delay)


## 2) Read and parse robots.txt (basic)

We will parse:

- `Disallow:` paths
- `Crawl-delay:`

This is a **simplified** parser suitable for teaching.


In [ ]:
def fetch_robots_txt(base_url: str) -> str:
    robots_url = urljoin(base_url, "robots.txt")
    r = requests.get(robots_url, timeout=20)
    r.raise_for_status()
    return r.text


def parse_robots_txt(robots_text: str, user_agent: str = "*"):
    disallow = []
    crawl_delay = None
    active_agent = None

    for raw in robots_text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        key, _, val = line.partition(":")
        key = key.strip().lower()
        val = val.strip()

        if key == "user-agent":
            active_agent = val
        elif key == "disallow" and active_agent in (user_agent, "*"):
            if val:
                disallow.append(val)
        elif key == "crawl-delay" and active_agent in (user_agent, "*"):
            try:
                crawl_delay = float(val)
            except ValueError:
                pass

    return {
        "disallow": disallow,
        "crawl_delay": crawl_delay,
    }


robots_text = fetch_robots_txt(BASE_URL)
robots_rules = parse_robots_txt(robots_text)
robots_rules

## 3) URL normalisation + allow/deny checks


In [ ]:
def normalise_url(base_url: str, href: str) -> str | None:
    if not href:
        return None
    # Join relative links
    abs_url = urljoin(base_url, href)
    # Remove fragments (#...)
    abs_url, _frag = urldefrag(abs_url)
    return abs_url


def is_same_site(url: str, base_url: str) -> bool:
    return urlparse(url).netloc == urlparse(base_url).netloc


def is_allowed_by_robots(url: str, base_url: str, disallow_paths: list[str]) -> bool:
    parsed = urlparse(url)
    path = parsed.path or "/"
    for dis in disallow_paths:
        # Disallow is path-prefix matching in this simplified model
        if path.startswith(dis):
            return False
    return True


## 4) Extraction helpers (cards)

We extract `div.sh-card` blocks and parse for tutor/listing IDs.


In [ ]:
def extract_card_texts(page) -> list[str]:
    cards = page.locator("div.sh-card")
    out = []
    for i in range(cards.count()):
        t = cards.nth(i).inner_text().strip()
        if t:
            out.append(t)
    return out


def parse_tutor_card(card_text: str) -> dict | None:
    # Require an ID like TUT-001 somewhere
    m_id = re.search(r"\bID:\s*(TUT-\d{3})\b", card_text)
    if not m_id:
        return None
    tutor_id = m_id.group(1)

    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title = lines[0]
    m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)\/h$", title)
    if not m:
        m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)", title)
    name = m.group("name").strip() if m else title
    price = int(m.group("price")) if m and m.groupdict().get("price") else None

    headline = None
    subjects = None
    languages = None
    for ln in lines[1:]:
        if ln.lower().startswith("subjects:"):
            subjects = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("languages:"):
            languages = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("id:"):
            continue
        else:
            if headline is None:
                headline = ln

    return {
        "id": tutor_id,
        "name": name,
        "price_eur_per_hour": price,
        "headline": headline,
        "subjects": subjects,
        "languages": languages,
        "source_url": None,
    }


def parse_listing_card(card_text: str) -> dict | None:
    m_id = re.search(r"\bID:\s*(LIST-\d{4})\b", card_text)
    if not m_id:
        return None
    listing_id = m_id.group(1)

    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title_line = lines[0]
    m = re.match(r"^(?P<title>.+?)\s*·\s*€(?P<rent>\d+)\/mo$", title_line)
    title = m.group("title").strip() if m else title_line
    rent = int(m.group("rent")) if m and m.groupdict().get("rent") else None

    listing_type = None
    area = None
    m2 = re.match(r"^(?P<type>.+?)\s+in\s+(?P<area>.+)$", title)
    if m2:
        listing_type = m2.group("type").strip()
        area = m2.group("area").strip()

    min_stay_months = None
    deposit_eur = None
    for ln in lines[1:]:
        if "min stay" in ln.lower():
            ms = re.search(r"min stay:\s*(\d+)\s*months", ln, flags=re.I)
            if ms:
                min_stay_months = int(ms.group(1))
            dep = re.search(r"deposit:\s*€(\d+)", ln, flags=re.I)
            if dep:
                deposit_eur = int(dep.group(1))

    return {
        "id": listing_id,
        "title": title,
        "type": listing_type,
        "area": area,
        "rent_eur_per_month": rent,
        "min_stay_months": min_stay_months,
        "deposit_eur": deposit_eur,
        "source_url": None,
    }


## 5) The crawler (BFS)

Key concepts:

- **Frontier (queue)**: URLs to visit
- **Visited set**: avoid revisiting
- **Politeness**: delay between requests
- **Scope control**: same host only

During each visit, we:

1. Collect new internal links
2. Extract any tutor/listing cards present on the page


In [ ]:
def crawl_site(base_url: str, max_pages: int = 50, user_agent: str = "*"):
    robots_text = fetch_robots_txt(base_url)
    rules = parse_robots_txt(robots_text, user_agent=user_agent)
    disallow = rules["disallow"]
    delay = rules["crawl_delay"] if rules["crawl_delay"] is not None else DEFAULT_DELAY

    frontier = deque([base_url])
    visited = set()

    tutors = {}
    listings = {}
    page_log = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        while frontier and len(visited) < max_pages:
            url = frontier.popleft()
            if url in visited:
                continue
            if not is_same_site(url, base_url):
                continue
            if not is_allowed_by_robots(url, base_url, disallow):
                continue

            t0 = time.time()
            try:
                page.goto(url, wait_until="networkidle", timeout=45000)
                # small extra sleep to allow streamlit widgets to settle
                time.sleep(0.8)

                # Extract links
                links = page.locator("a").all()
                new_links = 0
                for a in links:
                    href = a.get_attribute("href")
                    nxt = normalise_url(base_url, href) if href else None
                    if not nxt:
                        continue
                    if not is_same_site(nxt, base_url):
                        continue
                    if nxt in visited:
                        continue
                    if not is_allowed_by_robots(nxt, base_url, disallow):
                        continue
                    frontier.append(nxt)
                    new_links += 1

                # Extract cards
                cards = extract_card_texts(page)
                tutor_count = 0
                listing_count = 0
                for c in cards:
                    t = parse_tutor_card(c)
                    if t and t["id"] not in tutors:
                        t["source_url"] = url
                        tutors[t["id"]] = t
                        tutor_count += 1
                    l = parse_listing_card(c)
                    if l and l["id"] not in listings:
                        l["source_url"] = url
                        listings[l["id"]] = l
                        listing_count += 1

                visited.add(url)
                page_log.append({
                    "url": url,
                    "new_links_found": new_links,
                    "tutors_found": tutor_count,
                    "listings_found": listing_count,
                    "queue_size": len(frontier),
                    "elapsed_s": round(time.time() - t0, 2),
                })

            except Exception as e:
                visited.add(url)
                page_log.append({
                    "url": url,
                    "error": str(e),
                    "queue_size": len(frontier),
                })

            # Politeness delay
            time.sleep(delay)

        browser.close()

    return {
        "visited": visited,
        "page_log": pd.DataFrame(page_log),
        "tutors": pd.DataFrame(list(tutors.values())).sort_values("id"),
        "listings": pd.DataFrame(list(listings.values())).sort_values("id"),
        "robots": robots_text,
        "delay": delay,
    }


## 6) Run the crawler


In [ ]:
result = crawl_site(BASE_URL, max_pages=MAX_PAGES)

print("Visited pages:", len(result["visited"]))
print("Tutors collected:", len(result["tutors"]))
print("Listings collected:", len(result["listings"]))

result["page_log"].head(10)

## 7) Inspect the extracted dataframes


In [ ]:
tutors_df = result["tutors"].reset_index(drop=True)
listings_df = result["listings"].reset_index(drop=True)

display(tutors_df.head(10))
display(listings_df.head(10))

## 8) Save to CSV


In [ ]:
tutors_csv_path = "campusnest_crawled_tutors.csv"
listings_csv_path = "campusnest_crawled_listings.csv"
log_csv_path = "campusnest_crawl_log.csv"

tutors_df.to_csv(tutors_csv_path, index=False)
listings_df.to_csv(listings_csv_path, index=False)
result["page_log"].to_csv(log_csv_path, index=False)

(tutors_csv_path, listings_csv_path, log_csv_path)